In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


In [2]:
import pandas as pd
from src.utils.general import load_json

In [3]:
ROOT = "./models/text"

In [4]:
models = list(os.listdir(f"{ROOT}/node"))

raw_metrics = dict()

# 1. Separate (Node + Edge)
for model_setting in models:
    setting, model = model_setting.split("_")
    for dataset in os.listdir(f"{ROOT}/node/{model_setting}"):
        for split_type in os.listdir(f"{ROOT}/node/{model_setting}/{dataset}"):
            node_p = f"{ROOT}/node/{model_setting}/{dataset}/{split_type}/test_metrics.json"
            edge_p = f"{ROOT}/edge/{model_setting}/{dataset}/{split_type}/test_metrics.json"
            if os.path.exists(node_p) and os.path.exists(edge_p):
                node_metrics = load_json(node_p)
                edge_metrics = load_json(edge_p)
                raw_metrics[(setting, model, "separate", dataset, split_type)] = {
                    **{("Node", m): v for m, v in node_metrics.items()},
                    **{("Edge", m): v for m, v in edge_metrics.items()},
                }

# 2. Complete
if os.path.exists(f"{ROOT}/complete"):
    for model_setting in os.listdir(f"{ROOT}/complete"):
        setting, model = model_setting.split("_")
        for dataset in os.listdir(f"{ROOT}/complete/{model_setting}"):
            for split_type in os.listdir(f"{ROOT}/complete/{model_setting}/{dataset}"):
                comp_p = f"{ROOT}/complete/{model_setting}/{dataset}/{split_type}/test_metrics.json"
                if os.path.exists(comp_p):
                    comp_metrics = load_json(comp_p)
                    node_metrics = comp_metrics.get("node", {})
                    edge_metrics = comp_metrics.get("rel") or comp_metrics.get("edge", {})
                    raw_metrics[(setting, model, "complete", dataset, split_type)] = {
                        **{("Node", m): v for m, v in node_metrics.items()},
                        **{("Edge", m): v for m, v in edge_metrics.items()},
                    }

df = pd.DataFrame.from_dict(raw_metrics, orient="index")
df.index.names = ["setting", "model", "approach", "dataset", "split"]
df.columns.names = ["task", "metric"]


In [5]:
df

task                                                  Node            \
metric                                                  f1 precision   
setting   model      approach dataset  split                           
shared    bge        separate dataset2 inductive  0.410955  0.412473   
                                       hybrid     0.255127  0.277451   
                              dataset1 inductive  0.225326  0.253181   
                                       hybrid     0.097901  0.102763   
          roberta    separate dataset2 inductive  0.345703  0.351199   
                                       hybrid     0.268089  0.296092   
                              dataset1 inductive  0.176746  0.187120   
                                       hybrid     0.076871  0.094488   
dedicated modernbert separate dataset2 inductive  0.123023  0.206322   
                                       hybrid     0.178566  0.211518   
                              dataset1 inductive  0.068129  0.106122   
                                       hybrid     0.053054  0.057890   
shared    modernbert separate dataset2 inductive  0.238938  0.322889   
                                       hybrid     0.197844  0.228078   
                              dataset1 inductive  0.182666  0.218918   
                                       hybrid     0.070702  0.087043   
dedicated roberta    separate dataset2 inductive  0.244386  0.283957   
                                       hybrid     0.220794  0.248677   
                              dataset1 inductive  0.112529  0.148214   
                                       hybrid     0.069572  0.074032   
          bert       separate dataset2 inductive  0.208359  0.243209   
                                       hybrid     0.205161  0.254624   
                              dataset1 inductive  0.119029  0.152578   
                                       hybrid     0.070865  0.079460   
shared    bert       separate dataset2 inductive  0.329187  0.315181   
                                       hybrid     0.234913  0.262176   
                              dataset1 inductive  0.184956  0.191949   
                                       hybrid     0.094261  0.102761   
dedicated bge        separate dataset2 inductive  0.217484  0.266647   
                                       hybrid     0.219247  0.239128   
                              dataset1 inductive  0.101490  0.133218   
                                       hybrid     0.068209  0.072906   
shared    bge        complete dataset2 inductive  0.208626  0.222299   
                                       hybrid     0.149284  0.173700   
                              dataset1 inductive  0.312168  0.357466   
                                       hybrid     0.086904  0.105607   
          roberta    complete dataset2 inductive  0.259976  0.273171   
                                       hybrid     0.172935  0.206083   
                              dataset1 inductive  0.281727  0.315347   
                                       hybrid     0.062713  0.077921   
dedicated modernbert complete dataset2 hybrid     0.033747  0.045983   
                              dataset1 inductive  0.106270  0.122085   
                                       hybrid     0.022558  0.025979   
shared    modernbert complete dataset2 hybrid     0.137061  0.174105   
                              dataset1 inductive  0.218205  0.288409   
                                       hybrid     0.051963  0.067775   
dedicated roberta    complete dataset2 hybrid     0.099661  0.114515   
                              dataset1 inductive  0.154963  0.197480   
                                       hybrid     0.036937  0.039072   
          bert       complete dataset2 hybrid     0.068418  0.108280   
                              dataset1 inductive  0.224800  0.263004   
                                       hybrid     0.023833  0.031755   
shared    bert       complete dataset2 i

# DATASET 1

In [6]:
df.xs(("dataset1", "hybrid"), level=["dataset", "split"]).sort_index()

task                               Node                          Edge  \
metric                               f1 precision    recall        f1   
setting   model      approach                                           
dedicated bert       complete  0.023833  0.031755  0.051411  0.001774   
                     separate  0.070865  0.079460  0.142520  0.018045   
          bge        complete  0.042153  0.056828  0.061887  0.002612   
                     separate  0.068209  0.072906  0.143659  0.014336   
          modernbert complete  0.022558  0.025979  0.038772  0.000189   
                     separate  0.053054  0.057890  0.094773  0.006773   
          roberta    complete  0.036937  0.039072  0.059018  0.000190   
                     separate  0.069572  0.074032  0.135373  0.008609   
shared    bert       complete  0.056085  0.063650  0.091650  0.003464   
                     separate  0.094261  0.102761  0.167455  0.032075   
          bge        complete  0.086904  0.105607  0.113656  0.002565   
                     separate  0.097901  0.102763  0.175392  0.031933   
          modernbert complete  0.051963  0.067775  0.067578  0.003001   
                     separate  0.070702  0.087043  0.110046  0.015960   
          roberta    complete  0.062713  0.077921  0.090491  0.001991   
                     separate  0.076871  0.094488  0.119026  0.031028   

task                                               
metric                        precision    recall  
setting   model      approach                      
dedicated bert       complete  0.001573  0.006711  
                     separate  0.019030  0.023731  
          bge        complete  0.002475  0.004038  
                     separate  0.014820  0.021309  
          modernbert complete  0.000098  0.005720  
                     separate  0.007744  0.008593  
          roberta    complete  0.000098  0.003696  
                     separate  0.009756  0.014791  
shared    bert       complete  0.003120  0.009434  
                     separate  0.032845  0.048415  
          bge        complete  0.003447  0.006395  
                     separate  0.032230  0.044808  
          modernbert complete  0.004189  0.008099  
                     separate  0.017224  0.023313  
          roberta    complete  0.002905  0.005684  
                     separate  0.030886  0.045337

In [7]:
df.xs(("dataset1", "inductive"), level=["dataset", "split"]).sort_index()

task                               Node                          Edge  \
metric                               f1 precision    recall        f1   
setting   model      approach                                           
dedicated bert       complete  0.224800  0.263004  0.252719  0.000000   
                     separate  0.119029  0.152578  0.158943  0.063864   
          bge        complete  0.169088  0.213823  0.185605  0.007246   
                     separate  0.101490  0.133218  0.187642  0.052675   
          modernbert complete  0.106270  0.122085  0.134172  0.004831   
                     separate  0.068129  0.106122  0.111396  0.022151   
          roberta    complete  0.154963  0.197480  0.173189  0.000000   
                     separate  0.112529  0.148214  0.164697  0.040285   
shared    bert       complete  0.254908  0.278489  0.283920  0.014706   
                     separate  0.184956  0.191949  0.312875  0.128538   
          bge        complete  0.312168  0.357466  0.340269  0.010107   
                     separate  0.225326  0.253181  0.374558  0.166743   
          modernbert complete  0.218205  0.288409  0.236029  0.007194   
                     separate  0.182666  0.218918  0.275075  0.097168   
          roberta    complete  0.281727  0.315347  0.330297  0.007246   
                     separate  0.176746  0.187120  0.307114  0.154467   

task                                               
metric                        precision    recall  
setting   model      approach                      
dedicated bert       complete  0.000000  0.000000  
                     separate  0.081357  0.061628  
          bge        complete  0.007246  0.007246  
                     separate  0.060526  0.051091  
          modernbert complete  0.003623  0.007246  
                     separate  0.031847  0.020061  
          roberta    complete  0.000000  0.000000  
                     separate  0.058573  0.035424  
shared    bert       complete  0.014706  0.014706  
                     separate  0.151515  0.125967  
          bge        complete  0.014599  0.009037  
                     separate  0.191514  0.170412  
          modernbert complete  0.007194  0.007194  
                     separate  0.119136  0.091649  
          roberta    complete  0.007246  0.007246  
                     separate  0.176802  0.154369

# DATASET 2

In [8]:
df.xs(("dataset2", "hybrid"), level=["dataset", "split"]).sort_index()

task                               Node                          Edge  \
metric                               f1 precision    recall        f1   
setting   model      approach                                           
dedicated bert       complete  0.068418  0.108280  0.107841  0.001293   
                     separate  0.205161  0.254624  0.350645  0.025151   
          bge        complete  0.068918  0.123385  0.105021  0.010929   
                     separate  0.219247  0.239128  0.375341  0.019550   
          modernbert complete  0.033747  0.045983  0.076761  0.000000   
                     separate  0.178566  0.211518  0.299141  0.009394   
          roberta    complete  0.099661  0.114515  0.143384  0.019482   
                     separate  0.220794  0.248677  0.350992  0.042414   
shared    bert       complete  0.183361  0.237899  0.210652  0.019331   
                     separate  0.234913  0.262176  0.356474  0.068784   
          bge        complete  0.149284  0.173700  0.177954  0.021752   
                     separate  0.255127  0.277451  0.382772  0.063872   
          modernbert complete  0.137061  0.174105  0.180557  0.042531   
                     separate  0.197844  0.228078  0.331462  0.063475   
          roberta    complete  0.172935  0.206083  0.215219  0.032035   
                     separate  0.268089  0.296092  0.422190  0.092241   

task                                               
metric                        precision    recall  
setting   model      approach                      
dedicated bert       complete  0.000785  0.003663  
                     separate  0.031533  0.033228  
          bge        complete  0.010929  0.010929  
                     separate  0.020264  0.024096  
          modernbert complete  0.000000  0.000000  
                     separate  0.021213  0.010295  
          roberta    complete  0.018951  0.024031  
                     separate  0.048943  0.056107  
shared    bert       complete  0.018818  0.024621  
                     separate  0.084022  0.076684  
          bge        complete  0.025635  0.030561  
                     separate  0.085949  0.078119  
          modernbert complete  0.040089  0.058824  
                     separate  0.076564  0.076001  
          roberta    complete  0.036155  0.048513  
                     separate  0.103972  0.117465

In [9]:
df.xs(("dataset2", "inductive"), level=["dataset", "split"]).sort_index()

task                               Node                          Edge  \
metric                               f1 precision    recall        f1   
setting   model      approach                                           
dedicated bert       separate  0.208359  0.243209  0.304287  0.032228   
          bge        separate  0.217484  0.266647  0.248341  0.019800   
          modernbert separate  0.123023  0.206322  0.218863  0.000000   
          roberta    separate  0.244386  0.283957  0.391655  0.011008   
shared    bert       complete  0.255746  0.278694  0.408410  0.012836   
                     separate  0.329187  0.315181  0.575303  0.062544   
          bge        complete  0.208626  0.222299  0.320189  0.023127   
                     separate  0.410955  0.412473  0.569679  0.078573   
          modernbert separate  0.238938  0.322889  0.365117  0.060982   
          roberta    complete  0.259976  0.273171  0.443132  0.018784   
                     separate  0.345703  0.351199  0.602493  0.084578   

task                                               
metric                        precision    recall  
setting   model      approach                      
dedicated bert       separate  0.038734  0.033028  
          bge        separate  0.024861  0.022743  
          modernbert separate  0.000000  0.000000  
          roberta    separate  0.018964  0.009144  
shared    bert       complete  0.024952  0.012659  
                     separate  0.079090  0.075610  
          bge        complete  0.034230  0.041771  
                     separate  0.096832  0.109369  
          modernbert separate  0.081555  0.080398  
          roberta    complete  0.027875  0.028490  
                     separate  0.088000  0.117296